# Vina Scoring Pipeline BATCH

Objective: Take sorted SMILES strings of top small molecule candidates from LISARDD and PDB for target protein and produce a Vina scores for each of them and compare the order to our own binding afffinities

In [2]:
!pip install -q vina meeko biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.8/280.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 14.1 MB/s eta 0:00:00


In [1]:
###### Load inputs (Ligand SMILES and Protein PDB ID)

#ligand_smiles = "C1[C@H]2[C@@H]([C@@H](S1)CCCCC(=O)O)NC(=O)N2"  # Example: Biotin
#protein_pdb_id = "1SWB"  # Example: Streptavidin (unbound)
#protein_pdb_id = "1SWC" #APO-CORE-STREPTAVIDIN AT PH 4.5

#protein_pdb_id = "3RY1" #Wild-type core streptavidin at atomic resolution


ligand_smiles = "C[C@@]12[C@@H]([C@@H](C[C@@H](O1)N3C4=CC=CC=C4C5=C6C(=C7C8=CC=CC=C8N2C7=C53)CNC6=O)NC)OC"  # Example: Staurosporine
protein_pdb_id = "1HCK"  # Example: HUMAN CYCLIN-DEPENDENT KINASE 2 (should be unbound)

# ligand_smiles = "C[C@@]1(C(=O)N2[C@H](C(=O)N3CCC[C@H]3[C@@]2(O1)O)CC4=CC=CC=C4)NC(=O)[C@H]5CN([C@@H]6CC7=CNC8=CC=CC(=C78)C6=C5)C"  # Example: Ergotamine
# protein_pdb_id = "4IVA"  # Example: HUMAN CYCLIN-DEPENDENT KINASE 2 (should be unbound)





In [3]:
##### Download PDB and Process Receptor

from Bio.PDB import PDBList

# Download PDB file
pdbl = PDBList()
pdb_path = pdbl.retrieve_pdb_file(protein_pdb_id, pdir='.', file_format='pdb')

print('Downloaded PDB:', pdb_path)



Downloaded PDB: ./pdb1hck.ent


In [4]:
##### Prepare Ligand with Meeko

import meeko
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdmolfiles


# Prepare ligand molecule
mol = Chem.MolFromSmiles(ligand_smiles)
mol = Chem.AddHs(mol)  # Protonate
rdmolfiles.MolToPDBFile(mol, "ligand.pdb")

AllChem.EmbedMolecule(mol, AllChem.ETKDG())  # Generate 3D structure

# Meeko preparation
prep = meeko.MoleculePreparation()
prep.prepare(mol)

# Generate ligand pdbqt as string
ligand_pdbqt_str = prep.write_pdbqt_string()

print('Ligand prepared.')



Ligand prepared.


/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


In [5]:
##### Prepare Receptor PDBQT

!apt-get install -y openbabel

#!obabel pdb1swb.ent -O 1SWB_clean.pdbqt --addh #this is for streptavidin




Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 4 newly installed, 0 to remove and 34 not upgraded.
Need to get 3,903 kB of archives.
After this operation, 16.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,231 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openbabel amd64 3.1.1+dfsg-6ubuntu5 [128 kB]
Fetched 3,903 kB in 0s (21.5 MB/s)
Selecting previously unselected package libinchi1.
(Reading database ... 126101 files and direc

In [6]:


# Modular Receptor PDBQT Preparation
import os

def prepare_receptor_pdbqt(pdb_id, pdb_path):
    """
    Prepare receptor: download PDB if needed, clean, convert to PDBQT.
    Args:
        pdb_id (str): 4-letter PDB ID (e.g., "1SWB")
        pdb_path (str): Path to the downloaded PDB file
    Returns:
        receptor_pdbqt_path (str): Path to prepared receptor PDBQT file
    """
    pdbqt_filename = f"{pdb_id}_clean.pdbqt"

    # Convert PDB ➔ PDBQT
    !obabel "{pdb_path}" -O "{pdbqt_filename}" --addh

    # Clean the receptor PDBQT (remove ROOT, BRANCH if needed)
    cleaned_pdbqt_filename = f"{pdb_id}_clean_rigid.pdbqt"
    clean_receptor_pdbqt(pdbqt_filename, cleaned_pdbqt_filename)

    return cleaned_pdbqt_filename

# Cleaning function (same as before)
def clean_receptor_pdbqt(input_file, output_file):
    """
    Removes ligand-specific tags from receptor PDBQT.
    """
    with open(input_file, 'r') as infile:
        lines = infile.readlines()

    cleaned_lines = [line for line in lines if not (
        line.startswith('ROOT') or
        line.startswith('BRANCH') or
        line.startswith('ENDBRANCH') or
        line.startswith('ENDROOT') or
        line.startswith('TORSDOF')
    )]

    with open(output_file, 'w') as outfile:
        outfile.writelines(cleaned_lines)

# # Example Usage
# receptor_pdbqt_path = prepare_receptor_pdbqt(protein_pdb_id, pdb_path)

# print('Prepared receptor PDBQT saved to:', receptor_pdbqt_path)


In [7]:
##### Set up Docking

from vina import Vina

def run_docking(receptor_pdbqt_path, ligand_pdbqt_str, center, box_size, exhaustiveness=32, n_poses=5):
    """
    Set up and run AutoDock Vina docking given a prepared receptor and ligand.

    Args:
        receptor_pdbqt_path (str): Path to receptor .pdbqt
        ligand_pdbqt_str (str): Ligand PDBQT string from Meeko
        center (list): [center_x, center_y, center_z] docking box center
        box_size (list): [size_x, size_y, size_z] docking box size
        exhaustiveness (int): Vina exhaustiveness
        n_poses (int): Number of poses to generate
    Returns:
        vina_object: Vina object after docking (you can call v.energies(), v.poses(), etc.)
    """
    v = Vina(sf_name='vina')

    v.set_receptor(receptor_pdbqt_path)
    v.set_ligand_from_string(ligand_pdbqt_str)

    v.compute_vina_maps(center=center, box_size=box_size)

    v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)

    print('Docking complete.')

    return v




In [8]:
def compute_center_and_box(receptor_pdbqt_path, margin=5.0):
    """
    Computes center and box size automatically from a receptor PDBQT file.

    Args:
        receptor_pdbqt_path (str): Path to receptor PDBQT file.
        margin (float): Extra padding added to each box dimension (Å).

    Returns:
        center (list): [center_x, center_y, center_z]
        box_size (list): [size_x, size_y, size_z]
    """
    xs, ys, zs = [], [], []

    with open(receptor_pdbqt_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                xs.append(float(line[30:38]))
                ys.append(float(line[38:46]))
                zs.append(float(line[46:54]))

    if not xs:
        raise ValueError("No ATOM entries found in receptor PDBQT!")

    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    min_z, max_z = min(zs), max(zs)

    center_x = (min_x + max_x) / 2
    center_y = (min_y + max_y) / 2
    center_z = (min_z + max_z) / 2

    size_x = (max_x - min_x) + margin
    size_y = (max_y - min_y) + margin
    size_z = (max_z - min_z) + margin

    return [center_x, center_y, center_z], [size_x, size_y, size_z]


In [9]:
# Assuming you already have:
# - receptor_pdbqt_path from prepare_receptor_pdbqt()
# - ligand_pdbqt_str from Meeko prep
# - center, box_size known

receptor_pdbqt_path = prepare_receptor_pdbqt(protein_pdb_id, pdb_path)

# Now automatically calculate center/box
center, box_size = compute_center_and_box(receptor_pdbqt_path, margin=5.0)

# Proceed with docking
v = run_docking(receptor_pdbqt_path, ligand_pdbqt_str, center, box_size)


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ./pdb1hck.ent)

1 molecule converted
Docking complete.


In [10]:
print(v)

Receptor (rigid): 1HCK_clean_rigid.pdbqt
Receptor (flex): None
Ligands: REMARK SMILES CN[C@@H]1C[C@H]2O[C@@](C)([C@@H]1OC)n1c3ccccc3c3c4c(c5c6ccccc6n2c5c31)C(=O)NC4
REMARK SMILES IDX 8 1 7 2 9 3 3 4 4 5 5 6 6 7 29 8 28 9 27 10 26 11 25 12
REMARK SMILES IDX 24 13 23 14 22 15 21 16 20 17 19 18 18 19 17 20 16 21 15 22
REMARK SMILES IDX 14 23 13 24 12 25 31 26 30 27 35 28 34 29 32 30 33 31 10 33
REMARK SMILES IDX 11 34 2 35 1 36
REMARK H PARENT 34 32 2 37
ROOT
ATOM      1  C   UNL     1      -2.139   1.487  -2.022  1.00  0.00     0.071 C 
ATOM      2  C   UNL     1      -1.510   0.550  -0.979  1.00  0.00     0.173 C 
ATOM      3  C   UNL     1      -2.544   0.302   0.070  1.00  0.00     0.188 C 
ATOM      4  C   UNL     1      -3.034  -1.123  -0.167  1.00  0.00     0.091 C 
ATOM      5  C   UNL     1      -1.812  -1.981   0.117  1.00  0.00     0.077 C 
ATOM      6  C   UNL     1      -0.896  -1.722  -1.071  1.00  0.00     0.224 C 
ATOM      7  O   UNL     1      -1.191  -0.583  -1.741  1.0

In [13]:
##### Extract Docking Scores

# Get binding energies
energies = v.energies()

print('Docking Results:')
for idx, energy_array in enumerate(energies):
    binding_energy = energy_array[0]  # Extract only ΔG
    print(f'Pose {idx+1}: Binding Energy = {binding_energy:.3f} kcal/mol')


Docking Results:
Pose 1: Binding Energy = -7.548 kcal/mol
Pose 2: Binding Energy = -7.521 kcal/mol
Pose 3: Binding Energy = -7.491 kcal/mol
Pose 4: Binding Energy = -7.463 kcal/mol
Pose 5: Binding Energy = -7.411 kcal/mol


In [14]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [15]:
import os
import torch
import pickle
import numpy as np

# # Reload best smiles
# with open('/content/drive/MyDrive/HG2G/Binding Affinity Score Pipelines (Validation)/best_smiles_sorted.pkl', 'rb') as f:
#     best_smiles_sorted = pickle.load(f)

# # Reload best scores
# best_binding_score_sorted = np.load('/content/drive/MyDrive/HG2G/Binding Affinity Score Pipelines (Validation)/best_binding_score_sorted.npy')


##### CDK2

# Reload best smiles
with open('/content/drive/MyDrive/HG2G/Binding Affinity Score Pipelines (Validation)/best_smiles_sorted_cdk2.pkl', 'rb') as f:
    best_smiles_sorted = pickle.load(f)

# Reload best scores
best_binding_score_sorted = np.load('/content/drive/MyDrive/HG2G/Binding Affinity Score Pipelines (Validation)/best_binding_score_sorted_cdk2.npy')


In [23]:
##### Subset to the top 10 best smiles and associated binding affinities

# Top 10 (lower binding score = better)
#top_n = 10

top10_smiles = best_smiles_sorted[40:]
top10_scores = best_binding_score_sorted[40:]

top20_smiles = best_smiles_sorted[30:]
top20_scores = best_binding_score_sorted[30:]


In [17]:
##### Batch docking wrapper

import pickle

def batch_dock_smiles(smiles_list, receptor_pdbqt_path, center, box_size,
                      exhaustiveness=8, n_poses=5, cache_file="vina_cache.pkl"):
    """
    Batch dock a list of SMILES, avoid repeats, and return Vina scores aligned to input order.

    Args:
        smiles_list (list of str): SMILES strings to dock
        receptor_pdbqt_path (str): Path to receptor PDBQT
        center, box_size (list): Docking box center and size
        cache_file (str): Path to cache file storing {SMILES: score}
    Returns:
        vina_scores (list of float): Best Vina scores per input SMILES
    """
    import os
    import meeko
    from rdkit import Chem
    from rdkit.Chem import AllChem

    # Load cache if exists
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            vina_cache = pickle.load(f)
    else:
        vina_cache = {}

    scores = []
    for smi in smiles_list:
        if smi in vina_cache:
            scores.append(vina_cache[smi])
            continue

        try:
            mol = Chem.MolFromSmiles(smi)
            mol = Chem.AddHs(mol)
            AllChem.EmbedMolecule(mol, AllChem.ETKDG())

            # Save ligand as PDBQT string using Meeko
            prep = meeko.MoleculePreparation()
            prep.prepare(mol)
            ligand_pdbqt_str = prep.write_pdbqt_string()

            # Run docking
            v = run_docking(receptor_pdbqt_path, ligand_pdbqt_str, center, box_size,
                            exhaustiveness=exhaustiveness, n_poses=n_poses)

            # Extract best score
            energy = v.energies()[0][0]  # best pose
        except Exception as e:
            print(f"[!] Docking failed for {smi[:20]}...: {e}")
            energy = None

        vina_cache[smi] = energy
        scores.append(energy)

    # Save updated cache
    with open(cache_file, 'wb') as f:
        pickle.dump(vina_cache, f)

    return scores


In [24]:
### Run batch docking

# Make sure receptor is prepared
receptor_pdbqt_path = prepare_receptor_pdbqt(protein_pdb_id, pdb_path)
center, box_size = compute_center_and_box(receptor_pdbqt_path, margin=5.0)

# Dock all sorted SMILES
vina_scores = batch_dock_smiles(top20_smiles, receptor_pdbqt_path, center, box_size)


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ./pdb1hck.ent)

1 molecule converted


/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.


In [19]:
##### Top 10 comparison between MGraphDTA PkD and Vina

import numpy as np

# Convert to numpy for sorting
vina_scores_np = np.array(vina_scores)
affinity_scores_np = np.array(top10_scores)

# Rankings (lower = better)
vina_ranks = np.argsort(vina_scores_np)        # Vina scores (lower = better)
affinity_ranks = np.argsort(-affinity_scores_np)  # pKd (higher = better)

# Compare alignment
print("Top 5 by Vina:")
for i in vina_ranks[:5]:
    print(f"{i}: SMILES: {top10_smiles[i][:30]}... Vina: {vina_scores[i]:.2f}, pKd: {top10_scores[i]:.2f}")

print("\nTop 5 by pKd:")
for i in affinity_ranks[:5]:
    print(f"{i}: SMILES: {top10_smiles[i][:30]}... pKd: {top10_scores[i]:.2f}, Vina: {vina_scores[i]:.2f}")


Top 5 by Vina:
8: SMILES: Cc1cc(C)c2c(c1)CN(c1ccccc1[N+]... Vina: -6.86, pKd: 9.51
0: SMILES: CC(C)Cc1ccccc1-c1ccccc1[N+](=O... Vina: -6.84, pKd: 7.93
9: SMILES: CN(C)c1c(C(F)F)cc2c(c1F)-c1c(c... Vina: -6.67, pKd: 10.98
5: SMILES: CC(=O)C1(C)CCCN(c2nc([NH2+][O-... Vina: -6.53, pKd: 8.16
2: SMILES: Cc1c([N+](=O)[O-])ccc(-c2ccccc... Vina: -6.14, pKd: 7.97

Top 5 by pKd:
9: SMILES: CN(C)c1c(C(F)F)cc2c(c1F)-c1c(c... pKd: 10.98, Vina: -6.67
8: SMILES: Cc1cc(C)c2c(c1)CN(c1ccccc1[N+]... pKd: 9.51, Vina: -6.86
7: SMILES: CNC(=O)c1cccc([N+](=O)[O-])c1-... pKd: 8.65, Vina: -5.93
6: SMILES: CNC(=O)c1c([N+](=O)[O-])ccc([N... pKd: 8.20, Vina: -5.75
5: SMILES: CC(=O)C1(C)CCCN(c2nc([NH2+][O-... pKd: 8.16, Vina: -6.53


In [25]:
##### Top 20 comparison between MGraphDTA PkD and Vina

import numpy as np

# Convert to numpy for sorting
vina_scores_np = np.array(vina_scores)
affinity_scores_np = np.array(top20_scores)

# Rankings (lower = better)
vina_ranks = np.argsort(vina_scores_np)        # Vina scores (lower = better)
affinity_ranks = np.argsort(-affinity_scores_np)  # pKd (higher = better)

# Compare alignment
print("Top 5 by Vina:")
for i in vina_ranks[:5]:
    print(f"{i}: SMILES: {top20_smiles[i][:30]}... Vina: {vina_scores[i]:.2f}, pKd: {top20_scores[i]:.2f}")

print("\nTop 5 by pKd:")
for i in affinity_ranks[:5]:
    print(f"{i}: SMILES: {top20_smiles[i][:30]}... pKd: {top20_scores[i]:.2f}, Vina: {vina_scores[i]:.2f}")


Top 5 by Vina:
0: SMILES: CNC(=O)Cc1ccccc1-c1ccc(C)c([N+... Vina: -7.26, pKd: 7.74
18: SMILES: Cc1cc(C)c2c(c1)CN(c1ccccc1[N+]... Vina: -6.86, pKd: 9.51
10: SMILES: CC(C)Cc1ccccc1-c1ccccc1[N+](=O... Vina: -6.84, pKd: 7.93
19: SMILES: CN(C)c1c(C(F)F)cc2c(c1F)-c1c(c... Vina: -6.67, pKd: 10.98
15: SMILES: CC(=O)C1(C)CCCN(c2nc([NH2+][O-... Vina: -6.53, pKd: 8.16

Top 5 by pKd:
19: SMILES: CN(C)c1c(C(F)F)cc2c(c1F)-c1c(c... pKd: 10.98, Vina: -6.67
18: SMILES: Cc1cc(C)c2c(c1)CN(c1ccccc1[N+]... pKd: 9.51, Vina: -6.86
17: SMILES: CNC(=O)c1cccc([N+](=O)[O-])c1-... pKd: 8.65, Vina: -5.93
16: SMILES: CNC(=O)c1c([N+](=O)[O-])ccc([N... pKd: 8.20, Vina: -5.75
15: SMILES: CC(=O)C1(C)CCCN(c2nc([NH2+][O-... pKd: 8.16, Vina: -6.53


In [20]:
print(vina_scores)

[np.float64(-6.84), np.float64(-5.814), np.float64(-6.139), np.float64(-5.951), np.float64(-6.121), np.float64(-6.531), np.float64(-5.753), np.float64(-5.927), np.float64(-6.863), np.float64(-6.667)]


In [21]:
import pandas as pd

# Full mapping from SMILES to Vina scores
smiles_to_vina_score = dict(zip(top10_smiles, vina_scores))


# Map top10_smiles to their Vina scores
top10_vina_scores = [smiles_to_vina_score[smi] for smi in top10_smiles]

# Now build the DataFrame
df_top10 = pd.DataFrame({
    'SMILES': top10_smiles,
    'Predicted_pKd': top10_scores,
    'Vina_Score_kcalmol': top10_vina_scores
})

# Display
display(df_top10)


,SMILES,Predicted_pKd,Vina_Score_kcalmol
0,CC(C)Cc1ccccc1-c1ccccc1[N+](=O)[O-],7.929533,-6.840
1,CNCc1c(-c2ccccc2[N+](=O)[O-])cccc1[N+](=O)[O-],7.973454,-5.814
2,Cc1c([N+](=O)[O-])ccc(-c2ccccc2)c1[N+](=O)[O-],7.974819,-6.139
3,Cc1cccc(-c2ccccc2[N+](=O)[O-])c1[N+](=O)[O-],8.017278,-5.951
4,CCc1cc(-c2ccccc2)c([N+](=O)[O-])cc1C,8.125389,-6.121
5,CC(=O)C1(C)CCCN(c2nc([NH2+][O-])c(-c3ccccc3)[n...,8.163728,-6.531
6,CNC(=O)c1c([N+](=O)[O-])ccc([N+](=O)[O-])c1-c1...,8.198138,-5.753
7,CNC(=O)c1cccc([N+](=O)[O-])c1-c1cc(C)ccc1C,8.645997,-5.927
8,Cc1cc(C)c2c(c1)CN(c1ccccc1[N+](=O)[O-])C2=O,9.512930,-6.863
9,CN(C)c1c(C(F)F)cc2c(c1F)-c1c(cccc1[NH+](C)[O-]...,10.981145,-6.667


In [26]:
###### Build df for top 20

import pandas as pd

# Full mapping from SMILES to Vina scores
smiles_to_vina_score = dict(zip(top20_smiles, vina_scores))


# Map top10_smiles to their Vina scores
top20_vina_scores = [smiles_to_vina_score[smi] for smi in top20_smiles]

# Now build the DataFrame
df_top20 = pd.DataFrame({
    'SMILES': top20_smiles,
    'Predicted_pKd': top20_scores,
    'Vina_Score_kcalmol': top20_vina_scores
})

# Display
display(df_top20)


,SMILES,Predicted_pKd,Vina_Score_kcalmol
0,CNC(=O)Cc1ccccc1-c1ccc(C)c([N+](=O)[O-])c1,7.736850,-7.258
1,NC(=O)c1cc(-c2ccccc2[N+](=O)[O-])c(F)cc1F,7.748655,-6.373
2,CNC(=O)c1ccccc1-c1ccccc1[N+](=O)[O-],7.750554,-5.985
3,NC(=O)c1c(-c2ccccc2)cccc1[N+](=O)[O-],7.763814,-6.322
4,CNC(=O)c1c([N+](=O)[O-])ccc([N+](=O)[O-])c1C(C)C,7.844104,-5.157
5,CNC(=O)c1c([N+](=O)[O-])ccc([N+](=O)[O-])c1-c1...,7.848252,-5.968
6,NC(=O)c1cc([N+](=O)[O-])ccc1-c1c(F)cccc1F,7.880224,-6.293
7,CNCc1c([N+](=O)[O-])ccc(-c2ccccc2)c1[N+](=O)[O-],7.893270,-6.260
8,CN(C)C(=O)c1cc([N+](=O)[O-])cc(-c2ccccc2)c1[N+...,7.903509,-5.943
9,Cc1c(C(=O)N(C)C)cc(-c2ccccc2)cc1[N+](=O)[O-],7.908206,-6.101


In [ ]:
##### Test random molecules (SMILES) with Vina

#Random sample
from rdkit import Chem
from rdkit.Chem import rdmolfiles
from rdkit.Chem import AllChem
from rdkit.Chem import rdmolops
import random

# Function to generate random valid molecules
def generate_random_smiles(n=10, min_atoms=8, max_atoms=20):
    random_smiles = []
    for _ in range(n):
        mol = Chem.RWMol()
        num_atoms = random.randint(min_atoms, max_atoms)
        for _ in range(num_atoms):
            atom = Chem.Atom(random.choice([6, 7, 8]))  # C, N, O
            mol.AddAtom(atom)

        # Randomly connect atoms
        for i in range(num_atoms-1):
            mol.AddBond(i, i+1, Chem.rdchem.BondType.SINGLE)

        # Sanitize
        try:
            final_mol = mol.GetMol()
            Chem.SanitizeMol(final_mol)
            smi = Chem.MolToSmiles(final_mol)
            random_smiles.append(smi)
        except:
            continue  # skip broken molecules

    return random_smiles

# Generate 10 random valid SMILES
random_smiles_list = generate_random_smiles(10)

# Show them
for idx, smi in enumerate(random_smiles_list):
    print(f"{idx+1}: {smi}")



1: NOONCOOOON
2: CCCNCCOCONCOO
3: NCOONCONONOCNOO
4: NCNNNNCOONONOCNN
5: CNONONOCONN
6: OCNOCNCOO
7: CNCNNOOOC
8: OCNNONNNCNNONNO
9: CCONOOCNCNCNOCNOOOC
10: CONNOOCNOO


In [ ]:
### Run batch docking on random sampels

# Make sure receptor is prepared
receptor_pdbqt_path = prepare_receptor_pdbqt(protein_pdb_id, pdb_path)
center, box_size = compute_center_and_box(receptor_pdbqt_path, margin=5.0)

# Dock all sorted SMILES
vina_scores_random = batch_dock_smiles(random_smiles_list, receptor_pdbqt_path, center, box_size)


*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ./pdb3ry1.ent)

1 molecule converted


/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Docking complete.
Docking complete.


/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:626: DeprecationWarning: MoleculePreparation.write_pdbqt_string() is deprecated in Meeko v0.5. Pass the MoleculeSetup instance to PDBQTWriterLegacy.write_string(). MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)
/usr/local/lib/python3.11/dist-packages/meeko/preparation.py:420: DeprecationWarning: MoleculePreparation.setup is deprecated in Meeko v0.5. MoleculePreparation.prepare() returns a list of MoleculeSetup instances.
  warnings.warn(msg, DeprecationWarning)


Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.
Docking complete.


In [ ]:
import pandas as pd

# Full mapping from SMILES to Vina scores
smiles_to_vina_score_random = dict(zip(random_smiles_list, vina_scores_random))


# Map top10_smiles to their Vina scores
#vina_scores_random_map = [smiles_to_vina_score[smi] for smi in random_smiles_list]

# Now build the DataFrame
df_random = pd.DataFrame({
    'SMILES': random_smiles_list,
    'Vina_Score_kcalmol': vina_scores_random
})

# Display
display(df_random)


,SMILES,Vina_Score_kcalmol
0,NOONCOOOON,-4.611
1,CCCNCCOCONCOO,-4.855
2,NCOONCONONOCNOO,-5.296
3,NCNNNNCOONONOCNN,-5.591
4,CNONONOCONN,-4.867
5,OCNOCNCOO,-5.143
6,CNCNNOOOC,-4.592
7,OCNNONNNCNNONNO,-6.188
8,CCONOOCNCNCNOCNOOOC,-4.176
9,CONNOOCNOO,-4.988
